# BERT 模型的理解與實作
### 以 Fine-grained emotion classification 為例
Fine-grained 指的是 label 集的粒度很細，標籤可能同時包含angry、furious等語意相近的答案，類別的數量也較多；與之相對的概念 Coarse-grained 就像是正負情緒分類，由於只要分兩類，難度相較於 Fine-grained 來說低了不少。

本次的手作課程使用 Empathetic Dialogs 資料集，包含了 Speaker 與 Listener 之間的多輪對話，以及一句描述當下情境的句子。類別共有 32 種，是一個難度頗高的細粒度情緒分類 benchmark dataset。

在**基礎實作課程**中，我們的**目標是將純文字的原始資料轉成 BERT 可以接受的格式**，並將 BERT fine-tune 在我們的資料集上。

In [ ]:
# 安裝必要的套件
from IPython.display import clear_output
!pip install datasets
!pip install transformers
clear_output()

In [ ]:
import torch
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModel
import pprint

### 讀取原始數據並處理

In [ ]:
# 讀取資料集
train = pd.read_csv('https://github.com/HsiaoEn/nlp-transformer/releases/download/empatheticdialogues_dataset/emo_train.csv')
valid = pd.read_csv('https://github.com/HsiaoEn/nlp-transformer/releases/download/empatheticdialogues_dataset/emo_valid.csv')
clear_output()

train.head(10)

,conv_id,utterance_idx,context,prompt,speaker_idx,utterance,selfeval,tags
0,hit:0_conv:1,1,sentimental,I remember going to the fireworks with my best...,1,I remember going to see the fireworks with my ...,5|5|5_2|2|5,NaN
1,hit:0_conv:1,2,sentimental,I remember going to the fireworks with my best...,0,Was this a friend you were in love with_comma_...,5|5|5_2|2|5,NaN
2,hit:0_conv:1,3,sentimental,I remember going to the fireworks with my best...,1,This was a best friend. I miss her.,5|5|5_2|2|5,NaN
3,hit:0_conv:1,4,sentimental,I remember going to the fireworks with my best...,0,Where has she gone?,5|5|5_2|2|5,NaN
4,hit:0_conv:1,5,sentimental,I remember going to the fireworks with my best...,1,We no longer talk.,5|5|5_2|2|5,NaN
5,hit:0_conv:1,6,sentimental,I remember going to the fireworks with my best...,0,Oh was this something that happened because of...,5|5|5_2|2|5,NaN
6,hit:1_conv:2,1,afraid,i used to scare for darkness,2,it feels like hitting to blank wall when i se...,4|3|4_3|5|5,NaN
7,hit:1_conv:2,2,afraid,i used to scare for darkness,3,Oh ya? I don't really see how,4|3|4_3|5|5,NaN
8,hit:1_conv:2,3,afraid,i used to scare for darkness,2,dont you feel so.. its a wonder,4|3|4_3|5|5,NaN
9,hit:1_conv:2,4,afraid,i used to scare for darkness,3,I do actually hit blank walls a lot of times b...,4|3|4_3|5|5,NaN


In [ ]:
# 查看類別標籤與類別數量
print(f'Sentiment Lables: {train.context.unique()}')
print(f'Label Numbers:{len(train.context.unique())}')

Sentiment Lables: ['sentimental' 'afraid' 'proud' 'faithful' 'terrified' 'joyful' 'angry'
 'sad' 'jealous' 'grateful' 'prepared' 'embarrassed' 'excited' 'annoyed'
 'lonely' 'ashamed' 'guilty' 'surprised' 'nostalgic' 'confident' 'furious'
 'disappointed' 'caring' 'trusting' 'disgusted' 'anticipating' 'anxious'
 'hopeful' 'content' 'impressed' 'apprehensive' 'devastated']
Label Numbers:32


In [ ]:
# 原始標籤轉換為 id
label2idx = {'sad': 0, 'trusting': 1, 'terrified': 2, 'caring': 3, 'disappointed': 4,
             'faithful': 5, 'joyful': 6, 'jealous': 7, 'disgusted': 8, 'surprised': 9,
             'ashamed': 10, 'afraid': 11, 'impressed': 12, 'sentimental': 13,
             'devastated': 14, 'excited': 15, 'anticipating': 16, 'annoyed': 17, 'anxious': 18,
             'furious': 19, 'content': 20, 'lonely': 21, 'angry': 22, 'confident': 23,
             'apprehensive': 24, 'guilty': 25, 'embarrassed': 26, 'grateful': 27,
             'hopeful': 28, 'proud': 29, 'prepared': 30, 'nostalgic': 31}

idx2label = {v: k for k, v in label2idx.items()}

從這份資料集的任務目標來看，我們要將每一段 conversation 視為一筆要被判斷情緒的資料。

可利用「條件判斷」，從 Pandas DataFrame 中篩選要查看的資料

In [ ]:
train[train['conv_id'] == 'hit:0_conv:1']

,conv_id,utterance_idx,context,prompt,speaker_idx,utterance,selfeval,tags
0,hit:0_conv:1,1,sentimental,I remember going to the fireworks with my best...,1,I remember going to see the fireworks with my ...,5|5|5_2|2|5,NaN
1,hit:0_conv:1,2,sentimental,I remember going to the fireworks with my best...,0,Was this a friend you were in love with_comma_...,5|5|5_2|2|5,NaN
2,hit:0_conv:1,3,sentimental,I remember going to the fireworks with my best...,1,This was a best friend. I miss her.,5|5|5_2|2|5,NaN
3,hit:0_conv:1,4,sentimental,I remember going to the fireworks with my best...,0,Where has she gone?,5|5|5_2|2|5,NaN
4,hit:0_conv:1,5,sentimental,I remember going to the fireworks with my best...,1,We no longer talk.,5|5|5_2|2|5,NaN
5,hit:0_conv:1,6,sentimental,I remember going to the fireworks with my best...,0,Oh was this something that happened because of...,5|5|5_2|2|5,NaN


In [ ]:
# 查看特定對話中的 "utterance" 欄位
train[train['conv_id'] == 'hit:0_conv:1']['utterance']

,utterance
0,I remember going to see the fireworks with my ...
1,Was this a friend you were in love with_comma_...
2,This was a best friend. I miss her.
3,Where has she gone?
4,We no longer talk.
5,Oh was this something that happened because of...


除了「條件判斷式」，也可以利用 Pandas 的 `groupby` 將不同的 conv_id 分群，以便後續查看、應用

對 DataFrame 使用 groupby 之後，會回傳一個 groupby 物件 `pandas.api.typing.DataFrameGroupBy`

我們無法直接查看 groupby 物件的內容，若要查看 groupby 物件的內容，有以下幾種方法:
- 進行聚合 (Aggregation) 運算
- 迭代 (Iterate) groupby 物件
- 使用 `.get_group()` 取得單一一個群組

[Group by: split-apply-combine](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html)

In [ ]:
# 使用 get_group 取得 hit:0_conv:1 群組的資料
train.groupby('conv_id', sort=False).get_group('hit:0_conv:1')

,conv_id,utterance_idx,context,prompt,speaker_idx,utterance,selfeval,tags
0,hit:0_conv:1,1,sentimental,I remember going to the fireworks with my best...,1,I remember going to see the fireworks with my ...,5|5|5_2|2|5,NaN
1,hit:0_conv:1,2,sentimental,I remember going to the fireworks with my best...,0,Was this a friend you were in love with_comma_...,5|5|5_2|2|5,NaN
2,hit:0_conv:1,3,sentimental,I remember going to the fireworks with my best...,1,This was a best friend. I miss her.,5|5|5_2|2|5,NaN
3,hit:0_conv:1,4,sentimental,I remember going to the fireworks with my best...,0,Where has she gone?,5|5|5_2|2|5,NaN
4,hit:0_conv:1,5,sentimental,I remember going to the fireworks with my best...,1,We no longer talk.,5|5|5_2|2|5,NaN
5,hit:0_conv:1,6,sentimental,I remember going to the fireworks with my best...,0,Oh was this something that happened because of...,5|5|5_2|2|5,NaN


In [ ]:
# 查看 utterance 欄位
train.groupby('conv_id', sort=False).get_group('hit:0_conv:1')['utterance']

,utterance
0,I remember going to see the fireworks with my ...
1,Was this a friend you were in love with_comma_...
2,This was a best friend. I miss her.
3,Where has she gone?
4,We no longer talk.
5,Oh was this something that happened because of...


In [ ]:
# group 物件可進行迭代
for name, group in train.groupby('conv_id', sort=False):
    pprint.pp(f'Name: {name}')
    pprint.pp(f'Group Prompt: {group['prompt']}')
    pprint.pp(f'Group Utterance: {group['utterance']}')
    break

'Name: hit:0_conv:1'
('Group Prompt: 0    I remember going to the fireworks with my best...\n'
 '1    I remember going to the fireworks with my best...\n'
 '2    I remember going to the fireworks with my best...\n'
 '3    I remember going to the fireworks with my best...\n'
 '4    I remember going to the fireworks with my best...\n'
 '5    I remember going to the fireworks with my best...\n'
 'Name: prompt, dtype: object')
('Group Utterance: 0    I remember going to see the fireworks with my ...\n'
 '1    Was this a friend you were in love with_comma_...\n'
 '2                  This was a best friend. I miss her.\n'
 '3                                  Where has she gone?\n'
 '4                                   We no longer talk.\n'
 '5    Oh was this something that happened because of...\n'
 'Name: utterance, dtype: object')


In [ ]:
# 處理原始資料集所需要的函數
def process_data(data, label2idx):
    # 這個函數需要接收原始資料(data)及標籤名稱和id的對應(label2idx)，
    # 回傳清理乾淨的樣本(text)及對應的標籤 id (label)
    text, label = [], []
    for name, group_df in data.groupby('conv_id', sort=False):
            # 取出 prompt 再結合 utterance 對話內容，組合成一份對話段落
            conversation = ''
            conversation += group_df['prompt'].values[0] + ' '
            conversation += '[SEP]'.join(group_df['utterance'])

            # 在這裡可以做資料的清洗或前處理，可以根據你的觀察來決定如何處理
            conversation = conversation.replace('_comma_', ',')
            text.append(conversation)
            # 同時也需要將標籤從名稱 (例如 angry) 轉為 id (22)
            label.append(label2idx.get(group_df['context'].values[0]))

    return text, label

# 處理原始資料集
train_text, train_label = process_data(train, label2idx)
valid_text, valid_label = process_data(valid, label2idx)

In [ ]:
# 比對一下整理過後的資料
pprint.pp(f'Text: {train_text[0]}')
pprint.pp(f'Label: {train_label[0]}')

('Text: I remember going to the fireworks with my best friend. There was a lot '
 'of people, but it only felt like us in the world. I remember going to see '
 'the fireworks with my best friend. It was the first time we ever spent time '
 'alone together. Although there was a lot of people, we felt like the only '
 'people in the world.[SEP]Was this a friend you were in love with, or just a '
 'best friend?[SEP]This was a best friend. I miss her.[SEP]Where has she '
 'gone?[SEP]We no longer talk.[SEP]Oh was this something that happened because '
 'of an argument?')
'Label: 13'


In [ ]:
print("訓練樣本數：", len(train_text))
print("驗證樣本數：", len(valid_text))

訓練樣本數： 19533
驗證樣本數： 2770


## 使用 Hugging Face **AutoTokenizer** 對文本進行 Tokenization 處理

Hugging Face `AutoTokenizer` 會自動讀取您想使用的「模型名稱」，並載入與該模型配對的正確 Tokenizer。例如:

```{.sh}
from transformers import AutoTokenizer

text = 'I have a new GPU!'

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#    padding=True: 將句子填充到統一長度
#    truncation=True: 將過長的句子截斷
#    return_tensors="pt": 返回 PyTorch Tensors (如果是 TF 用 "tf")

inputs = tokenizer(
    text,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print(inputs)
```

Tokenizer 輸出的結果是字典:

{\
'input_ids': tensor([[  101,  1045,  2031,  1037,  2047, 14246,  2226,   999,   102]]), \
'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), \
'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]]) \
}

tokenizer 也有 tokenize 方法，執行該方法可以將所傳入的文字切分成 token，例如:

```{.sh}
tokenizer.tokenize(text)

# ["i", "have", "a", "new", "gp", "##u", "!"]
```

In [ ]:
# 指定模型，使用 AutoTokenizer 載入該模型的 Tokenizer
MODEL = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
print(tokenizer.model_input_names)

['input_ids', 'token_type_ids', 'attention_mask']


## 動手寫一個自定義的 Dataset Class
`torch.utils.data.Dataset` 是一個代表資料集的抽象類別 (abstract class)。所自訂的資料集應該繼承 (inherit) `Dataset` 並覆寫 (override) 以下方法：

- `__len__`：如此一來 `len(dataset)` 就能回傳資料集的大小。
- `__getitem__`：用以支援索引 (indexing)，使得 `dataset[i]` 可以被用來取得第 i 個樣本 (sample)。

[torch.utils.data](https://docs.pytorch.org/docs/stable/data.html)

In [ ]:
class EmotionDataset(Dataset):
    # 讓 max_len 成為參數，更靈活
    def __init__(self, text, label, tokenizer, max_len=150):
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len  # 儲存 max_len

    def __len__(self):
        return len(self.text)

    # 定義回傳一筆資料時要做的事
    def __getitem__(self, idx):
        # 取得原始文本和標籤
        text = self.text[idx]
        label = self.label[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,         # 自動截斷 (處理 "錯誤的 Truncation" 問題)
            padding='max_length',    # 自動填充 (處理 "没有 Padding" 問題)
            max_length=self.max_len, # 指定最大長度
            return_tensors='pt'      # 直接回傳 PyTorch Tensors
        )
        # --------------------------------------------------------------------

        # encoding 是一個字典，包含 'input_ids', 'token_type_ids', 'attention_mask'
        # 每個 value 都是一個 tensor，shape 是 [1, max_len]
        # 我們使用 .squeeze(0) 將 [1, max_len] 轉為 [max_len]

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'token_type_ids': encoding['token_type_ids'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 初始化剛剛定義的 Dataset
trainset = EmotionDataset(train_text, train_label, tokenizer=tokenizer)
validset = EmotionDataset(valid_text, valid_label, tokenizer=tokenizer)

In [ ]:
trainset[0]

{'input_ids': tensor([  101,  1045,  3342,  2183,  2000,  1996, 16080,  2007,  2026,  2190,
          2767,  1012,  2045,  2001,  1037,  2843,  1997,  2111,  1010,  2021,
          2009,  2069,  2371,  2066,  2149,  1999,  1996,  2088,  1012,  1045,
          3342,  2183,  2000,  2156,  1996, 16080,  2007,  2026,  2190,  2767,
          1012,  2009,  2001,  1996,  2034,  2051,  2057,  2412,  2985,  2051,
          2894,  2362,  1012,  2348,  2045,  2001,  1037,  2843,  1997,  2111,
          1010,  2057,  2371,  2066,  1996,  2069,  2111,  1999,  1996,  2088,
          1012,   102,  2001,  2023,  1037,  2767,  2017,  2020,  1999,  2293,
          2007,  1010,  2030,  2074,  1037,  2190,  2767,  1029,   102,  2023,
          2001,  1037,  2190,  2767,  1012,  1045,  3335,  2014,  1012,   102,
          2073,  2038,  2016,  2908,  1029,   102,  2057,  2053,  2936,  2831,
          1012,   102,  2821,  2001,  2023,  2242,  2008,  3047,  2138,  1997,
          2019,  6685,  1029,   102,   

In [ ]:
# 選擇第一個樣本出來，看看原始 input 是怎麼被轉換成 BERT 相容的格式的
sample_idx = 0

# 將原始文本拿出做比較
sample_text, sample_label = train_text[sample_idx], train_label[sample_idx]
first_item = trainset[0]

# 將 tokens_tensor 還原成文本
tokens = tokenizer.convert_ids_to_tokens(first_item['input_ids'].tolist())
combined_text = " ".join(tokens)

pprint.pp(f"""[原始文本]
句子：{sample_text}
分類：{sample_label}

===============

[從 Dataset tensor 還原的句子]
{combined_text}
""")

('[原始文本]\n'
 '句子：I remember going to the fireworks with my best friend. There was a lot of '
 'people, but it only felt like us in the world. I remember going to see the '
 'fireworks with my best friend. It was the first time we ever spent time '
 'alone together. Although there was a lot of people, we felt like the only '
 'people in the world.[SEP]Was this a friend you were in love with, or just a '
 'best friend?[SEP]This was a best friend. I miss her.[SEP]Where has she '
 'gone?[SEP]We no longer talk.[SEP]Oh was this something that happened because '
 'of an argument?\n'
 '分類：13\n'
 '\n'
 '===============\n'
 '\n'
 '[從 Dataset tensor 還原的句子]\n'
 '[CLS] i remember going to the fireworks with my best friend . there was a '
 'lot of people , but it only felt like us in the world . i remember going to '
 'see the fireworks with my best friend . it was the first time we ever spent '
 'time alone together . although there was a lot of people , we felt like the '
 'only people in the worl

In [ ]:
# 初始化每次回傳一批樣本的 DataLoader
BATCH_SIZE = 32
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validloader = DataLoader(validset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# 檢查 DataLoader 是否運作正常
first_batch = next(iter(trainloader))

print(f"Batch 的 Keys: {first_batch.keys()}")
# 檢查 Tensors 的形狀 (Shape)
# 預期形狀: [BATCH_SIZE, max_len] (也就是 [16, 150])
print(f"input_ids 的形狀: {first_batch['input_ids'].shape}")
print(f"attention_mask 的形狀: {first_batch['attention_mask'].shape}")

# 預期形狀: [BATCH_SIZE] (也就是 [16])
print(f"labels 的形狀: {first_batch['labels'].shape}")

Batch 的 Keys: dict_keys(['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
input_ids 的形狀: torch.Size([32, 150])
attention_mask 的形狀: torch.Size([32, 150])
labels 的形狀: torch.Size([32])


## 資料準備完畢，載入預訓練好的語言模型
Hugging face 整理並實現了許多語言模型，而且提供訓練好的參數可以直接載入。有了厲害的預訓練語言模型，機器就能更好的理解我們輸入的人類語言，也就是把離散的文字輸入轉化成考慮了上下文而產生的連續向量，具備這樣的武器，後續無論是要應對什麼樣的問題，難度都大大的降低了許多。

想要載入訓練好的模型很容易，只要指定要載入的模型名稱(e.g., `bert-base-uncased`, `roberta-large` 等)，就可以直接從 hugging face 提供的 hub 將模型拉進來。各種變形和模型的名稱可以在 https://huggingface.co/transformers/pretrained_models.html 這個頁面找到。

In [ ]:
from transformers import AutoModelForSequenceClassification

BERT_model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=32)
clear_output()

## 開始訓練模型

[Packing and Unpacking Arguments in Python](https://www.geeksforgeeks.org/python/packing-and-unpacking-arguments-in-python/)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, trainloader, validloader, epochs, lr=1e-5):

    # Optimizer 使用 AdamW 效果更好
    optimizer = AdamW(model.parameters(), lr=lr)
    model = model.to(device)

    for epoch in range(epochs):

        # --- 訓練模式 ---
        model.train()
        total_train_loss = 0.0
        train_pred, train_labels = [], []


        # "batch" 是一個字典: {'input_ids': ..., 'attention_mask': ..., ...}
        for batch in tqdm(trainloader):

            # 將字典中的所有 Tensors 移到 device
            batch = {k: v.to(device) for k, v in batch.items()}

            # 清除梯度
            optimizer.zero_grad()

            # 使用 **batch 將字典解包後送入模型
            # HHF 模型會自動抓取 'input_ids', 'attention_mask' 等...
            outputs = model(**batch)

            loss, logits = outputs['loss'], outputs['logits']
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            train_pred.extend(predictions.cpu().tolist()) # .cpu() 是一個好習慣
            train_labels.extend(batch['labels'].cpu().tolist()) # 從 batch 中取 labels

        train_loss = total_train_loss / len(trainloader)
        train_acc = accuracy_score(train_pred, train_labels)
        train_f1 = f1_score(train_pred, train_labels, average='micro')
        print(f'Epoch {epoch+1}/{epochs} | train | acc: {train_acc:.04f}, f1: {train_f1:.04f}, loss: {train_loss:.04f}')

        # --- 驗證模式 ---
        model.eval() # 驗證前必須加 model.eval()
        total_valid_loss = 0.0
        valid_pred, valid_labels = [], []

        with torch.no_grad(): # 驗證時不計算梯度

            # 驗證迴圈 (同訓練迴圈)
            for batch in tqdm(validloader):

                batch = {k: v.to(device) for k, v in batch.items()}

                outputs = model(**batch)
                loss, logits = outputs['loss'], outputs['logits']

                total_valid_loss += loss.item()

                predictions = torch.argmax(logits, dim=-1)
                valid_pred.extend(predictions.cpu().tolist())
                valid_labels.extend(batch['labels'].cpu().tolist())

        valid_loss = total_valid_loss / len(validloader)

        # 變數是 valid_labels (複數)
        valid_acc = accuracy_score(valid_pred, valid_labels)
        valid_f1 = f1_score(valid_pred, valid_labels, average='micro')
        print(f'Epoch {epoch+1}/{epochs} | valid | acc: {valid_acc:.04f}, f1: {valid_f1:.04f}, loss: {valid_loss:.04f}')
        print("-" * 50)

    return model

In [ ]:
BERT_model = train_model(BERT_model, trainloader, validloader, epochs = 3)

  0%|          | 0/611 [00:00<?, ?it/s]

Epoch 1/3 | train | acc: 0.2861, f1: 0.2861, loss: 2.7192


  0%|          | 0/87 [00:00<?, ?it/s]

Epoch 1/3 | valid | acc: 0.5047, f1: 0.5047, loss: 1.9280
--------------------------------------------------


  0%|          | 0/611 [00:00<?, ?it/s]

...

Epoch 2/3 | valid | acc: 0.5949, f1: 0.5949, loss: 1.3992
--------------------------------------------------


  0%|          | 0/611 [00:00<?, ?it/s]

Epoch 3/3 | train | acc: 0.6428, f1: 0.6428, loss: 1.1937


  0%|          | 0/87 [00:00<?, ?it/s]

Epoch 3/3 | valid | acc: 0.6144, f1: 0.6144, loss: 1.2552
--------------------------------------------------


## 儲存訓練好的模型
一般來說，我們應該在訓練的同時記錄最小的 validation loss 或最高的 validation accuracy，以此來決定儲存模型的時機，不過在這裡我們就直接儲存訓練完 3 個 epochs 後的參數。

In [ ]:
torch.save(BERT_model.state_dict(), 'bert.pt')

要將訓練好的模型讀取回來，做預測或繼續訓練也都是很方便的：

In [ ]:
BERT_model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=32)
BERT_model.load_state_dict(torch.load("bert.pt"))
BERT_model.to(device)
BERT_model.eval()
clear_output()

## 對測試集中的新樣本進行推論

In [ ]:
def inference(model, testloader):

    # 推論前必須切換到 eval() 模式
    model.eval()

    test_preds, test_labels = [], []

    # 驗證/推論時，不需要計算梯度
    with torch.no_grad():

        # data -> batch
        for batch in tqdm(testloader):

            # 將字典 Tensors 移到 device
            batch = {k: v.to(device) for k, v in batch.items()}

            # 使用 **batch 解包字典送入模型
            outputs = model(**batch)

            # HHF 模型會自動回傳 logits (即使 batch 中包含 labels)
            logits = outputs['logits']

            predictions = torch.argmax(logits, dim=-1)

            # 從 batch 中取得 labels
            test_preds.extend(predictions.cpu().tolist()) # 最好加上 .cpu() 再 .tolist()
            test_labels.extend(batch['labels'].cpu().tolist())

    test_acc = accuracy_score(test_preds, test_labels)
    test_f1 = f1_score(test_preds, test_labels, average='micro')

    print(f"Inference | acc: {test_acc:.04f}, f1: {test_f1:.04f}")
    return test_acc, test_f1, test_preds, test_labels

In [ ]:
valid_acc, valid_f1, bert_pred, valid_labels = inference(BERT_model, validloader)
print('======= Bert performance =======')
print(f'valid | acc: {valid_acc:.04f}, f1: {valid_f1:.04f}')

  0%|          | 0/87 [00:00<?, ?it/s]

Inference | acc: 0.6144, f1: 0.6144
======= Bert performance =======
valid | acc: 0.6144, f1: 0.6144
